# SnowHealth Demand Forecasting - Deployment & Monitoring
Deploy the tuned model for batch inference via warehouse compute and set up model drift monitoring with alerting.

In [ ]:
# Initialize
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
from model_utils import compute_metrics, create_prediction_df
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print("Session ready")

In [ ]:
# Load the registered model from Model Registry
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)

print(f"Model: {cfg.SNOWHEALTH_MODEL_NAME}")
print(f"Default version: {model_ref.default}")
for v in model_ref.versions():
    print(f"  {v.version_name}: {v.comment}")

mv = model_ref.version("tuned_v1")
print(f"\nLoaded version: tuned_v1")
print(f"Functions: {[f['name'] for f in mv.show_functions()]}")

In [ ]:
# ===== BATCH INFERENCE (Warehouse Compute) =====
# Load test data for batch prediction
df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES").to_pandas()
feature_cols = get_feature_columns(df, cfg.TARGET_COLUMN)

X = df[feature_cols].astype(float)
y = df[cfg.TARGET_COLUMN].astype(float)
_, X_test, _, y_test = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)
_, df_test_meta, _, _ = train_test_split(df, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)

# Run batch inference via Model Registry
predictions = mv.run(X_test)
print(f"Batch predictions generated: {len(predictions)} rows")
print(f"Sample predictions:\n{predictions.head(10)}")

In [ ]:
# Create prediction table with metadata for monitoring
pred_df = df_test_meta[['STORE_ID', 'SKU_ID', 'TRANSACTION_DATE']].copy().reset_index(drop=True)
pred_df['UNITS_SOLD'] = y_test.values
pred_df['PREDICTED_UNITS'] = predictions['UNITS_SOLD'].values
pred_df['RESIDUAL'] = pred_df['UNITS_SOLD'] - pred_df['PREDICTED_UNITS']
pred_df['ABS_PCT_ERROR'] = np.abs(pred_df['RESIDUAL']) / np.maximum(pred_df['UNITS_SOLD'], 1) * 100
pred_df['MODEL_VERSION'] = 'tuned_v1'
pred_df['PREDICTION_TIMESTAMP'] = pd.Timestamp.now()

# Save to Snowflake for monitoring
for col in pred_df.select_dtypes(include=['int8', 'int16', 'int32']).columns:
    pred_df[col] = pred_df[col].astype('int64')

snow_pred = session.create_dataframe(pred_df)
snow_pred.write.mode('overwrite').save_as_table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_PREDICTIONS")
print(f"Predictions saved: {len(pred_df)} rows")
print(f"Mean Absolute Error: {pred_df['RESIDUAL'].abs().mean():.2f}")
print(f"Mean Absolute % Error: {pred_df['ABS_PCT_ERROR'].mean():.1f}%")

In [ ]:
# Set up Model Monitor on the predictions table written above
monitor_fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_MONITOR"
source_fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_PREDICTIONS"

session.sql(f"DROP MODEL MONITOR IF EXISTS {monitor_fqn}").collect()
session.sql(f"""
CREATE MODEL MONITOR {monitor_fqn} WITH
    MODEL = {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{cfg.SNOWHEALTH_MODEL_NAME}
    VERSION = 'TUNED_V1'
    FUNCTION = 'PREDICT'
    SOURCE = {source_fqn}
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_UNITS')
    ACTUAL_SCORE_COLUMNS = ('UNITS_SOLD')
    SEGMENT_COLUMNS = ('STORE_ID')
""").collect()
print("Model Monitor created: SNOWHEALTH_DEMAND_MONITOR")
print(f"  Model:  {cfg.SNOWHEALTH_MODEL_NAME} v TUNED_V1")
print(f"  Source: SNOWHEALTH_DEMAND_PREDICTIONS (segmented by STORE_ID)")
print("  View in Snowsight: AI & ML > Models > SNOWHEALTH_DEMAND_FORECAST > Monitors")

In [ ]:
# SQL-based monitoring queries (work even without Model Monitor)
print("=" * 60)
print("MONITORING QUERIES (run periodically or via alerts)")
print("=" * 60)

# Calculate current model performance from predictions table
perf = session.sql(f"""
    SELECT
        MODEL_VERSION,
        COUNT(*) AS N_PREDICTIONS,
        ROUND(AVG(ABS(RESIDUAL)), 4) AS MAE,
        ROUND(SQRT(AVG(RESIDUAL * RESIDUAL)), 4) AS RMSE,
        ROUND(AVG(ABS_PCT_ERROR), 2) AS MAPE,
        ROUND(STDDEV(RESIDUAL), 4) AS RESIDUAL_STD,
        MIN(PREDICTION_TIMESTAMP) AS FIRST_PREDICTION,
        MAX(PREDICTION_TIMESTAMP) AS LAST_PREDICTION
    FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_PREDICTIONS
    GROUP BY MODEL_VERSION
""").to_pandas()
print("\nModel Performance Summary:")
print(perf.to_string(index=False))

# Check for prediction anomalies
anomalies = session.sql(f"""
    SELECT
        STORE_ID,
        COUNT(*) AS N_HIGH_ERROR,
        ROUND(AVG(ABS_PCT_ERROR), 1) AS AVG_PCT_ERROR
    FROM {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_PREDICTIONS
    WHERE ABS_PCT_ERROR > 30
    GROUP BY STORE_ID
    ORDER BY AVG_PCT_ERROR DESC
""").to_pandas()
print("\nHigh-Error Predictions by Store (>30% error):")
print(anomalies.to_string(index=False) if len(anomalies) > 0 else "  None detected")

In [ ]:
# Set up alerting for drift detection
try:
    session.sql(f"""
    CREATE OR REPLACE ALERT {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_DRIFT_ALERT
        WAREHOUSE = COMPUTE_WH
        SCHEDULE = 'USING CRON 0 8 * * * America/New_York'
    IF (EXISTS (
        SELECT 1
        FROM TABLE({cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_SQL_MONITOR!GET_MODEL_MONITOR_METRICS(
            METRIC_TYPE => 'RMSE',
            START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
            END_TIME => CURRENT_TIMESTAMP(),
            AGGREGATION_WINDOW => 'PT24H'
        ))
        WHERE VALUE > 6.0
    ))
    THEN
        CALL SYSTEM$SEND_EMAIL(
            'SNOWHEALTH_DRIFT_NOTIFICATION',
            'ml-ops@cvs.example.com',
            'SnowHealth Demand Forecast - Drift Alert',
            'Model drift detected: RMSE exceeds threshold of 6.0. Review model performance.'
        )
    """).collect()
    session.sql(f"ALTER ALERT {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_DRIFT_ALERT RESUME").collect()
    print("Drift alert created and resumed")
except Exception as e:
    print(f"Alert setup info: {e}")
    print("Note: Email notification integration may need to be configured separately.")
    print("The alert SQL pattern is ready for deployment.")

In [ ]:
# Verify deployment
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted
axes[0].scatter(pred_df['UNITS_SOLD'], pred_df['PREDICTED_UNITS'], alpha=0.3, s=10, color='#cc0000')
max_val = max(pred_df['UNITS_SOLD'].max(), pred_df['PREDICTED_UNITS'].max())
axes[0].plot([0, max_val], [0, max_val], 'k--', linewidth=1)
axes[0].set_xlabel('Actual Units')
axes[0].set_ylabel('Predicted Units')
axes[0].set_title('Actual vs Predicted', fontsize=13, fontweight='bold')

# Error distribution
axes[1].hist(pred_df['RESIDUAL'], bins=40, color='#cc0000', edgecolor='white')
axes[1].axvline(x=0, color='black', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_title('Prediction Error Distribution', fontsize=13, fontweight='bold')

# Error by store
store_errors = pred_df.groupby('STORE_ID')['ABS_PCT_ERROR'].mean().sort_values()
axes[2].barh(store_errors.index, store_errors.values, color='#cc0000')
axes[2].set_xlabel('Mean Absolute % Error')
axes[2].set_title('Error by Store', fontsize=13, fontweight='bold')

plt.suptitle('SnowHealth Demand Forecast - Deployment Validation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print("=" * 70)
print("SNOWHEALTH DEMAND FORECASTING - DEPLOYMENT & MONITORING COMPLETE")
print("=" * 70)
print(f"")
print(f"MODEL REGISTRY:")
print(f"  Model: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{cfg.SNOWHEALTH_MODEL_NAME}")
print(f"  Versions: baseline_v1, tuned_v1 (default)")
print(f"  Target: WAREHOUSE (batch inference ready)")
print(f"")
print(f"BATCH INFERENCE:")
print(f"  Predictions: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.SNOWHEALTH_DEMAND_PREDICTIONS")
print(f"  Rows: {len(pred_df):,}")
print(f"")
print(f"MONITORING:")
print(f"  Model Monitor: SNOWHEALTH_DEMAND_MONITOR")
print(f"  Source: SNOWHEALTH_DEMAND_PREDICTIONS")
print(f"  Alert: SNOWHEALTH_DEMAND_DRIFT_ALERT (daily at 8 AM ET)")
print(f"  Threshold: RMSE > 6.0 triggers notification")
print(f"")
print(f"EXPERIMENT TRACKING:")
print(f"  Experiment: {cfg.SNOWHEALTH_EXPERIMENT_NAME}")
print(f"  Runs: xgboost_baseline, lightgbm_baseline, xgboost_tuned_hpo")
print(f"")
print(f"FEATURE STORE:")
print(f"  Store: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}")
print(f"  Entity: {cfg.SNOWHEALTH_ENTITY_NAME}")
print(f"  Feature View: {cfg.SNOWHEALTH_FEATURE_VIEW_NAME} v{cfg.SNOWHEALTH_FEATURE_VIEW_VERSION}")
print("=" * 70)